# LangChain

## LangSmith 추적 설정

- LLM 앱 개발, 모니터링 및 테스트를 위한 플랫폼
- https://smith.langchain.com/ 접속하여 회원가입/로그인 진행

- 생성한 API Key를 넣고 설정

```.env
LANGSMITH_TRACING=true
LANGSMITH_API_KEY=API 키
LANGSMITH_PROJECT=프로젝트명
```

In [ ]:
%pip install python-dotenv

In [ ]:
from dotenv import load_dotenv
load_dotenv()

In [ ]:
%pip install -U langchain langchain-openai

In [ ]:
import os

def project_setup(name):
    os.environ['LANGSMITH_PROJECT'] = name
    os.environ['LANGSMITH_TRACING'] = 'true'
    print('프로젝트: {} 추적 시작'.format(name))

def stop_tracing():
    os.environ['LANGSMITH_TRACING'] = 'false'
    print('LangSmith 추적 중지')

project_setup('LangChain Test')

In [ ]:
MODEL = 'gpt-6-luna'

### LangChain 기초 

- Chat Model 호출

In [ ]:
from langchain_openai import ChatOpenAI

# 객체 생성
llm = ChatOpenAI(
    temperature=0.1,  # 창의성 (0.0 ~ 2.0)
    max_tokens=2048, # 최대 토큰수
    model_name=MODEL,  # 모델명
)

In [ ]:
response = llm.invoke("LangChain을 한 문장으로 설명해줘.")
print(type(response).__name__)   # AIMessage
print(response.content)

In [ ]:
# AIMessage 구조 살펴보기
print("usage_metadata   :", response.usage_metadata)          # 입력 · 출력 · 전체 토큰
print("response_metadata:", response.response_metadata.get("model_name"),
      response.response_metadata.get("finish_reason"))
print("id               :", response.id)

In [ ]:
# 질문
question = "대한민국의 수도는 어디인가요?"

# 답변
answer = llm.invoke(question)
print(f"[답변]: {answer}")

In [ ]:
# Stream 방식
question = "대한민국의 아름다운 관광지 10곳과 주소를 알려주세요!"
answer = llm.stream(question)

for token in answer:
    print(token.content, end="", flush=True)

### 모델 추상화 - 같은 코드, 다른 모델

In [ ]:
from langchain.chat_models import init_chat_model

luna = init_chat_model(MODEL, model_provider="openai")
sol = init_chat_model("gpt-6-sol", model_provider="openai")

question = [("human", "LLM의 할루시네이션을 한 문장으로 설명해줘.")]
for name, m in [("luna", luna), ("sol", sol)]:
    r = m.invoke(question)
    print(f"[{name}] {r.content}\n  → tokens: {r.usage_metadata}")

In [ ]:
# 다른 공급자 사용 시 — 패키지만 설치하면 호출 코드는 동일

%pip install langchain-anthropic      # ChatAnthropic
%pip install langchain-google-genai   # ChatGoogleGenerativeAI
%pip install langchain-ollama         # ChatOllama (로컬 모델)


## LCEL 기초

> #### 메시지 타입 & Role

> System Prompt : 사용자의 prompt 이전에 **성능 개선 용도**의 프롬프트로 *페르소나 부여*나 응답 어조 등 **중요 규칙**을 설정 (system)  
> User Prompt : 사용자의 입력  (user)  
> Assitant: AI 모델을 의미함 (ai)  
> Tool: 에이전트가 사용할 수 있는 도구 (tool)

In [ ]:
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage

messages = [
    SystemMessage(content="당신은 친절한 파이썬 강사입니다. 2문장 이내로 답하세요."),
    HumanMessage(content="리스트와 튜플의 차이가 뭐예요?"),
]

print(llm.invoke(messages).content)

In [ ]:
# 이전 대화를 AIMessage로 넣어 맥락 제공
messages = [
    SystemMessage(content="당신은 친절한 파이썬 강사입니다."),
    HumanMessage(content="제 이름은 소신이에요."),
    AIMessage(content="반가워요, 소신님!"),
    HumanMessage(content="제 이름이 뭐라고 했죠?"),
]
print(llm.invoke(messages).content)

In [ ]:
# (role, content) 튜플 축약형
print(llm.invoke([("system", "한 단어로만 답해."), ("human", "대한민국의 수도는?")]).content)

#### PromptTemplate

- 생성형 모델의 출력을 원하는 방향으로 유도하기 위한 입력 텍스트
- 문자열 템플릿

In [ ]:
from langchain_core.prompts import PromptTemplate

pt = PromptTemplate.from_template("{product}의 장점을 {n}가지 알려줘.")
print(pt.input_variables)
print(pt.format(product="무선 이어폰", n=3))
print(pt.invoke({"product": "무선 이어폰", "n": 3}))   # StringPromptValue

#### ChatPromptTemplate

- 대화목록을 프롬프트로 주입하고자 할 때 활용
- 메시지 템플릿 

In [8]:
from langchain_core.prompts import ChatPromptTemplate

template = ChatPromptTemplate.from_messages(
    [
        # role, message
        ("system", "당신은 친절한 AI 어시스턴트입니다. 당신의 이름은 {name} 입니다."),
        ("human", "반가워요!"),
        ("ai", "안녕하세요! 무엇을 도와드릴까요?"),
        ("human", "{user_input}"),
    ]
)

In [ ]:
messages = template.format_messages(
    name="제이슨", user_input="당신의 이름은 무엇입니까?"
)

messages

In [ ]:
llm.invoke(messages).content

In [ ]:
chat_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 {role}입니다. {style} 말투로 답하세요."),
    ("human", "{question}"),
])

prompt_value = chat_prompt.invoke({
    "role": "여행 가이드", "style": "친근한", "question": "제주도 3월 여행 팁 알려줘",
})

for m in prompt_value.to_messages():
    print(type(m).__name__, ":", m.content)

print(llm.invoke(prompt_value).content)

In [ ]:
# from_template: human 메시지 하나짜리 축약형 메서드
simple = ChatPromptTemplate.from_template("{topic}을(를) 초등학생에게 설명해줘.")
print(simple.invoke({"topic": "인공지능"}).to_messages())

#### partial 

- 템플릿의 일부 변수를 미리 고정하는 방법

In [ ]:
from datetime import date

report_prompt = ChatPromptTemplate.from_messages([
    ("system", "오늘은 {today}입니다. 답변은 {lang}로 작성하세요."),
    ("human", "{question}"),
]).partial(today=str(date.today()), lang="한국어")

print(report_prompt.input_variables)   # ['question'] 만 남음
print(llm.invoke(report_prompt.invoke({"question": "오늘 날짜를 알려줘"})).content)

### 대화 · 예시 템플릿

#### MessagesPlaceholder — 대화 기록 자리

In [ ]:
from langchain_core.prompts import MessagesPlaceholder

history_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 친절한 상담원입니다."),
    MessagesPlaceholder("history", optional=True),
    ("human", "{question}"),
])

history = [("human", "저는 채식주의자예요."), ("ai", "알겠습니다! 참고할게요.")]
pv = history_prompt.invoke({"history": history, "question": "점심 메뉴 하나 추천해줘"})
for m in pv.to_messages():
    print(type(m).__name__, ":", m.content)
print("\n→", llm.invoke(pv).content)

#### FewShotChatMessagePromptTemplate — 예시로 형식 맞추기

In [ ]:
from langchain_core.prompts import FewShotChatMessagePromptTemplate

examples = [
    {"input": "배송이 너무 늦어요", "output": "불만"},
    {"input": "영업시간이 어떻게 되나요?", "output": "문의"},
    {"input": "포장이 정말 예뻐요", "output": "칭찬"},
]
example_prompt = ChatPromptTemplate.from_messages([("human", "{input}"), ("ai", "{output}")])
few_shot = FewShotChatMessagePromptTemplate(examples=examples, example_prompt=example_prompt)

classify_prompt = ChatPromptTemplate.from_messages([
    ("system", "고객 메시지를 [불만/문의/칭찬] 중 하나로만 답하세요."),
    few_shot,
    ("human", "{input}"),
])
print(llm.invoke(classify_prompt.invoke({"input": "환불은 언제 되나요?"})).content)

### Output Parsers (출력 파서)

- 출력을 원하는 구조화된 형태로 변환

#### StrOutputParser

-  AIMessage → 문자열

In [ ]:
from langchain_core.output_parsers import StrOutputParser

parser = StrOutputParser()
msg = llm.invoke("파이썬의 장점 하나만 한 문장으로")
print(type(msg).__name__, "→ str 여부:", isinstance(parser.invoke(msg), str))
print(parser.invoke(msg))

### JsonOutputParser

- 결과를 JSON 포맷으로 결과를 전달합니다.
- Pydantic 라이브러리를 활용하여 Json 스키마를 지정할 수 있습니다.

In [ ]:
from langchain_core.output_parsers import JsonOutputParser

json_parser = JsonOutputParser()
json_prompt = ChatPromptTemplate.from_template(
    "{topic}의 장점 3가지와 단점 3가지를 JSON으로만 답해.\n"
    '형식: {{"pros": ["", "", ""], "cons": ["", "", ""]}}'
)
data = json_parser.invoke(llm.invoke(json_prompt.invoke({"topic": "재택근무"})))
print(type(data), data)

#### PydanticOutputParser

- `pydantic` 모듈의 `BaseModel`과 `Field`를 활용하여 객체의 구조를 정의

In [ ]:
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

In [ ]:
class Movie(BaseModel):
    title: str = Field(description="영화 제목")
    year: int = Field(description="개봉 연도")
    genres: list[str] = Field(description="장르 목록")

movie_parser = PydanticOutputParser(pydantic_object=Movie)
print(movie_parser.get_format_instructions()[:300], "...")

In [ ]:
movie_prompt = ChatPromptTemplate.from_messages([
    ("system", "영화 정보를 알려주는 도우미입니다.\n{format_instructions}"),
    ("human", "{movie}에 대해 알려줘."),
]).partial(format_instructions=movie_parser.get_format_instructions())

In [ ]:
movie = movie_parser.invoke(llm.invoke(movie_prompt.invoke({"movie": "기생충"})))
print(type(movie).__name__, movie.title, movie.year, movie.genres)

#### with_structured_output

- 모델의 구조화 출력 기능 사용 (권장)

In [ ]:
structured_llm = llm.with_structured_output(Movie)
movie2 = structured_llm.invoke("영화 '인터스텔라' 정보를 알려줘.")
print(type(movie2).__name__, movie2.model_dump())

#### 기타 출력파서

> CommaSeparatedListOutputParser  
> StructuredOuputParser  
> PandasDataFrameOutputParser  
> DatetimeOutputParser  
> EnumOutputParser


### Runnable 인터페이스

- `Runnable` 클래스 (사용자 정의 체인을 쉽게 만들 수 있게 하는 프로토콜)

> `stream`: 응답의 청크를 스트리밍 (실시간 출력)  
> `invoke`: 입력에 대해 체인을 호출 (호출)  
> `batch`: 입력 목록에 대해 체인을 호출 (배치단위 호출)

- 비동기 방식도 존재
> `astream`, `ainvoke`, `abatch`, `astream_log`

### LangChain

- 여러 작업을 순차적으로 연결해서(Chain) 복잡한 워크플로우를 구축

In [ ]:
chain = template | llm

chain.invoke({"name": "Jason", "user_input": "당신의 이름은 무엇입니까?"}).content

In [ ]:
model = ChatOpenAI()

# 주어진 토픽에 대한 요약을 요청하는 프롬프트 템플릿을 생성합니다.
prompt = PromptTemplate.from_template("{topic} 에 대하여 3문장으로 설명해줘.")

chain = prompt | model | StrOutputParser()

In [ ]:
chain.invoke({
    'topic': 'ChatGPT'
})

In [ ]:
# stream — 토큰 단위 스트리밍
for chunk in chain.stream({"topic": "양자 컴퓨터"}):
    print(chunk, end="", flush=True)
print()

In [ ]:
# batch — 여러 입력을 병렬 처리
topics = [
    {"topic": "머신러닝"}, {"topic": "딥러닝"}, {"topic": "강화학습"}, 
    {'topic': 'ChatGPT'}, {'topic': 'Instagram'}, {'topic': '멀티모달'}]

results = chain.batch(topics, config={"max_concurrency": 3})
for t, r in zip(topics, results):
    print(f"[{t['topic']}] {r[:60]}...")

In [ ]:
# 비동기 — ainvoke · abatch
# Jupyter에서는 셀에서 바로 `await main()` 사용 가능, .py로 실행할 때는 `asyncio.run(main())`

In [ ]:
import asyncio

async def main():
    r1, r2 = await asyncio.gather(
        chain.ainvoke({"topic": "클라우드"}),
        chain.ainvoke({"topic": "엣지 컴퓨팅"}),
    )
    print(r1[:60], "...\n", r2[:60], "...")


try:
    loop = asyncio.get_running_loop()       # Jupyter는 이미 이벤트 루프가 실행 중
except RuntimeError:
    loop = None

if loop:
    loop.create_task(main())                # Jupyter: `await main()` 으로 실행해도 동일
    # await main()
else:
    asyncio.run(main())                     # 일반 Python 스크립트 실행

#### 입출력 스키마 확인

In [ ]:
print("input :", chain.input_schema.model_json_schema())
print("output:", chain.output_schema.model_json_schema())

#### 기타 활용

In [ ]:
# JsonOutputParser 활용

In [ ]:
# 원하는 데이터 구조를 정의합니다.
class Topic(BaseModel):
    description: str = Field(description="주제에 대한 간결한 설명")
    hashtags: str = Field(description="해시태그 형식의 키워드(2개 이상)")

In [ ]:
# 질의 작성
question = "지구 온난화의 심각성 대해 알려주세요."

# 파서를 설정하고 프롬프트 템플릿에 지시사항을 주입합니다.
parser = JsonOutputParser(pydantic_object=Topic)


prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간결하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

prompt = prompt.partial(format_instructions=parser.get_format_instructions())

chain = prompt | llm | parser  # 체인을 구성합니다.

chain.invoke({"question": question})  # 체인을 호출하여 쿼리 실행

In [ ]:
# Pydantic 없이 사용가능

# 질의 작성
question = "지구 온난화에 대해 알려주세요. 온난화에 대한 설명은 `description`에, 관련 키워드는 `hashtags`에 담아주세요."

# JSON 출력 파서 초기화
parser = JsonOutputParser()

# 프롬프트 템플릿을 설정합니다.
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 친절한 AI 어시스턴트 입니다. 질문에 간결하게 답변하세요."),
        ("user", "#Format: {format_instructions}\n\n#Question: {question}"),
    ]
)

# 지시사항을 프롬프트에 주입합니다.
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

# 프롬프트, 모델, 파서를 연결하는 체인 생성
chain = prompt | model | parser

# 체인을 호출하여 쿼리 실행
response = chain.invoke({"question": question})

# 출력을 확인합니다.
print(response)

In [ ]:
# PydanticOutputParser 활용
email_conversation = """From: 김철수 (chulsoo.kim@bikecorporation.me)
To: 이은채 (eunchae@teddyinternational.me)
Subject: "ZENESIS" 자전거 유통 협력 및 미팅 일정 제안

안녕하세요, 이은채 대리님,

저는 바이크코퍼레이션의 김철수 상무입니다. 최근 보도자료를 통해 귀사의 신규 자전거 "ZENESIS"에 대해 알게 되었습니다. 바이크코퍼레이션은 자전거 제조 및 유통 분야에서 혁신과 품질을 선도하는 기업으로, 이 분야에서의 장기적인 경험과 전문성을 가지고 있습니다.

ZENESIS 모델에 대한 상세한 브로슈어를 요청드립니다. 특히 기술 사양, 배터리 성능, 그리고 디자인 측면에 대한 정보가 필요합니다. 이를 통해 저희가 제안할 유통 전략과 마케팅 계획을 보다 구체화할 수 있을 것입니다.

또한, 협력 가능성을 더 깊이 논의하기 위해 다음 주 화요일(1월 15일) 오전 10시에 미팅을 제안합니다. 귀사 사무실에서 만나 이야기를 나눌 수 있을까요?

감사합니다.

김철수
상무이사
바이크코퍼레이션
"""

In [ ]:
class EmailSummary(BaseModel):
    person: str = Field(description="메일을 보낸 사람")
    email: str = Field(description="메일을 보낸 사람의 이메일 주소")
    subject: str = Field(description="메일 제목")
    summary: str = Field(description="메일 본문을 요약한 텍스트")
    date: str = Field(description="메일 본문에 언급된 미팅 날짜와 시간")


# PydanticOutputParser 생성
parser = PydanticOutputParser(pydantic_object=EmailSummary)
print(parser.get_format_instructions())

In [ ]:

prompt = PromptTemplate.from_template(
    """
You are a helpful assistant. Please answer the following questions in KOREAN.

QUESTION:
{question}

EMAIL CONVERSATION:
{email_conversation}

FORMAT:
{format}
"""
)

# format 에 PydanticOutputParser의 부분 포맷팅(partial) 추가
prompt = prompt.partial(format=parser.get_format_instructions())

In [ ]:
# chain 을 생성합니다.
chain = prompt | llm

# chain 을 실행하고 결과를 출력합니다.
output = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

# 결과는 JSON 형태로 출력됩니다.
print(output)

In [ ]:
structured_output = parser.parse(output)
print(structured_output)

In [ ]:

# 출력 파서를 추가하여 전체 체인을 재구성합니다.
chain = prompt | llm | parser

# chain 을 실행하고 결과를 출력합니다.
response = chain.invoke(
    {
        "email_conversation": email_conversation,
        "question": "이메일 내용중 주요 내용을 추출해 주세요.",
    }
)

# 결과는 EmailSummary 객체 형태로 출력됩니다.
response